# DynaMate — Tutorial: Dynamic Multi-Agent Loop

This notebook walks through the complete **add → use → remove** lifecycle
of the DynaMate supervisor framework using a scientifically motivated example.

| Phase | Action |
|-------|--------|
| **1. Build** | Assemble the initial system and inspect the architecture |
| **2. Add** | Register a tool and create a dynamic agent via natural language |
| **3. Use** | Run a computation routed through the **Prompt Enhancer → Supervisor** pipeline |
| **4. Remove** | Delete the agent and tool, restoring the original architecture |

**Scientific context:** The Boltzmann thermal energy *kT* is the fundamental energy scale
in statistical mechanics and molecular simulation. We will register a `boltzmann_energy`
function as a tool, assign it to a dedicated `thermal_agent`, run a two-tool Arrhenius
calculation using the Prompt Enhancer, then cleanly remove both the agent and the tool.

---

### Framework overview

```
User prompt  (natural language — no tool/agent names required)
    │
    ▼
PromptEnhancer            ← lightweight LLM; reads live pool state and rewrites
    │                        the query with explicit routing hints
    ▼
Supervisor                ← LLM router; always reflects the current pool
    ├── tool_manager      ← registers tools, assigns them, adds/removes agents
    ├── shell_agent       ← runs shell commands (has terminal base tool)
    ├── compute_agent     ← general computation (starts empty)
    └── [dynamic agents]  ← created at runtime; persisted across sessions
              ▲
         AgentPool  (shared state)
         ├── _tool_registry   {name → StructuredTool}
         ├── _source_registry {name → source_code}  ← for persistence
         └── _agents          {name → {model, base_tools, extra_tools, ...}}
```

> **Package:** `dynamate` — built on LangGraph and `langgraph-supervisor`.
> Every pool mutation is written to `pool_state.json` immediately.

---
## 1 — Setup and System Initialization

DynaMate is assembled in **four ordered steps** to resolve the dependency between
the ToolManager (needs a pool reference) and the Supervisor (needs the ToolManager):

| Step | Call | Effect |
|------|------|--------|
| 1 | `PersistentAgentPoolWithSupervisor(...)` | Pool created; no supervisor yet |
| 2 | `pool.add_agent(...)` × 2 | `shell_agent` and `compute_agent` added |
| 3 | `build_tool_manager_v2(pool, model)` | ToolManager compiled, bound to the live pool |
| 4 | `pool.set_system_agents([tool_manager])` | **First supervisor build triggered** |

**Why `_is_dynamic=False` for the initial agents?**
Initial agents are always rebuilt from `main.py` on startup (they hold Python objects like
`ShellTool` that cannot be serialized). Setting `_is_dynamic=False` tells the persistent
pool to skip saving them to `pool_state.json`.

> **Isolation:** This tutorial writes to a temporary directory and does **not** affect
> your real `.dynamate/` session.

In [13]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import dotenv
dotenv.load_dotenv()

from IPython.display import display, Image,Markdown
from langchain_openai import ChatOpenAI
from langchain_community.tools import ShellTool
from dynamate import (
    PersistentAgentPoolWithSupervisor,
    PersistentSaver,
    PoolStore,
    PromptEnhancer,
    build_tool_manager_v2,
    pretty_print_messages,
    register_tools_from_prompt,
)

MODEL_NAME = 'gpt-4.1-mini'
model = ChatOpenAI(model=MODEL_NAME, temperature=0.0)

print(f'Model  : {MODEL_NAME}')
print('Imports: OK')

Model  : gpt-4.1-mini
Imports: OK


In [ ]:
# ── Temporary state directory (isolated from real .dynamate/) ─────────────────
TMP = tempfile.mkdtemp(prefix='dynamate_tutorial_')

SUPERVISOR_PROMPT = '''You are the Supervisor managing a pool of agents.
- tool_manager              : registers tools, assigns them to agents, and adds/removes agents.
- shell_agent               : runs shell commands and handles file-system tasks.
- compute_agent             : performs calculations with its dynamically assigned tools.
- [dynamic specialist agents]: created at runtime; equipped with domain tools.

Routing rules:
  * Add/register/assign/remove tools or agents -> tool_manager.
  * Shell or file-system tasks -> shell_agent.
  * Computation -> the specialist agent that has the required tool(s).
  * Python code (def statements) + add/register/use intent -> tool_manager
    (call register_tool_from_code — never shell_agent).
  * If no agent exists for the task, ask tool_manager to create one first.

Execution rules:
  * If you have all you need execute tasks immediately. 
  * When a specialist agent completes a calculation, report the full numerical
    result directly. Do not say "the agent is ready" or ask what to do next.
  * Assign work to one agent at a time.'''

# Step 1 — persistence layers
saver      = PersistentSaver(os.path.join(TMP, 'conversations'))
pool_store = PoolStore(os.path.join(TMP, 'pool_state.json'))

# Step 2 — create pool (no supervisor yet)
pool = PersistentAgentPoolWithSupervisor(
    supervisor_model=model,
    pool_store=pool_store,
    supervisor_prompt=SUPERVISOR_PROMPT,
    checkpointer=saver,
)

# Step 3 — add initial agents (_is_dynamic=False: not written to pool_state.json)
pool.add_agent(
    name='shell_agent',
    model=model,
    base_tools=[ShellTool()],
    system_prompt='You are a shell agent. Execute shell commands to answer requests.',
    _is_dynamic=False,
)
pool.add_agent(
    name='compute_agent',
    model=model,
    base_tools=[],
    system_prompt='You are a computation agent. Use your dynamically assigned tools to answer requests. Always call your tools and return the numerical result.',
    _is_dynamic=False,
)

# Step 4 — build ToolManager and trigger first supervisor build
tool_manager = build_tool_manager_v2(pool, model)
pool.set_system_agents([tool_manager])

# Step 5 — build PromptEnhancer (wraps the same model; queries pool live on each call)
enhancer = PromptEnhancer(model=model, pool=pool)

print('System ready.')
print(f'Agents   : {pool.list_agents()}')
print(f'Tools    : {pool.list_registered_tools() or "(none)"}')
print(f'Enhancer : {enhancer.__class__.__name__} ready')
print(f'State    : {TMP}')

System ready.
Agents   : ['shell_agent', 'compute_agent']
Tools    : (none)
Enhancer : PromptEnhancer ready
State    : /tmp/739186.1.long/dynamate_tutorial_i0ty60es


In [29]:
def print_pool_state(pool, label='POOL STATE'):
    'Print a concise snapshot of the current pool agents and their tools.'
    sep = '=' * 58
    print(f'\n{sep}')
    print(f'  {label}')
    print(sep)
    print(f'  Agents   : {pool.list_agents()}')
    print(f'  Registry : {pool.list_registered_tools() or "(empty)"}')
    print()
    for name in pool.list_agents():
        entry = pool._agents[name]
        base  = [t.name for t in entry['base_tools']]
        extra = [t.name for t in entry['extra_tools']]
        print(f'  [{name}]')
        print(f'    base tools    : {base or "(none)"}')
        print(f'    assigned tools: {extra or "(none)"}')
    print(sep)

In [30]:
print('── Initial Architecture ─────────────────────────────────────')
#display(Image(pool.supervisor.get_graph().draw_mermaid_png()))

mermaid_code = pool.supervisor.get_graph().draw_mermaid()
# Add a custom style for a node named 'supervisor'
mermaid_code += "\nclassDef supervisor fill:#f96,stroke:#333,stroke-width:4px;"
mermaid_code += "\nclass supervisor supervisor;"

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

print_pool_state(pool, 'INITIAL STATE')

── Initial Architecture ─────────────────────────────────────


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	tool_manager(tool_manager)
	shell_agent(shell_agent)
	compute_agent(compute_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	compute_agent --> supervisor;
	shell_agent --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> compute_agent;
	supervisor -.-> shell_agent;
	supervisor -.-> tool_manager;
	tool_manager --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

classDef supervisor fill:#f96,stroke:#333,stroke-width:4px;
class supervisor supervisor;
```


  INITIAL STATE
  Agents   : ['shell_agent', 'compute_agent']
  Registry : (empty)

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)


In [31]:
# ── Step A: register boltzmann_energy via a natural-language prompt ─────
# register_tools_from_prompt() is imported from dynamate.
# It uses an LLM to extract the function definition and calls
# pool.register_tool_from_code() directly — same as the direct API pattern:
#
#   result = pool.register_tool_from_code(BOLTZMANN_CODE)
#
# but the code is derived from free-text instead of a hard-coded string.

user_prompt_A = """
I have a Python function that computes the Boltzmann thermal energy.
Please add it to the system so I can use it later.

def boltzmann_energy(temperature_K: float) -> str:
    'Compute the Boltzmann thermal energy kT in eV for a given temperature in Kelvin. k_B = 8.617333e-5 eV/K.'
    kT = 8.617333e-5 * temperature_K
    return f'kT at {temperature_K:.1f} K = {kT:.6f} eV'
"""

result = register_tools_from_prompt(user_prompt_A, pool, model)
print(f'Registration : {result}')
print(f'Registry now : {pool.list_registered_tools()}')

Registration : Registered: boltzmann_energy
Registry now : ['boltzmann_energy']


In [28]:
# ── Verify the tool was registered correctly ─────────────────────────────
registered = pool.list_registered_tools()
print(f"Registry now : {registered}")
assert "boltzmann_energy" in registered, "boltzmann_energy not in registry!"

# Retrieve and call the live tool object to confirm it works
tool_obj = pool._tool_registry["boltzmann_energy"]
result = tool_obj.invoke({"temperature_K": 300.0})
print(f"Tool call    : boltzmann_energy(300 K) -> {result}")
print_pool_state(pool, "AFTER register_tools_from_prompt")

Registry now : ['boltzmann_energy', 'arrhenius_factor']
Tool call    : boltzmann_energy(300 K) -> kT at 300.0 K = 0.025852 eV

  AFTER register_tools_from_prompt
  Agents   : ['shell_agent', 'compute_agent']
  Registry : ['boltzmann_energy', 'arrhenius_factor']

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)


---
## 2 — Add: Register a Second Tool and Create a Dynamic Agent

In this phase we:
1. Register `arrhenius_factor` via a natural-language prompt (same pattern as Phase 1)
2. Ask the supervisor (through the PromptEnhancer) to create a `thermal_agent` and equip it with both tools

The user never names any internal framework objects — they just describe what they want.

In [29]:
import uuid

# One thread_id keeps conversation context across all supervisor calls in this tutorial
THREAD_ID = str(uuid.uuid4())
config = {"configurable": {"thread_id": THREAD_ID}}
print(f"Thread ID: {THREAD_ID}")

# ── Register arrhenius_factor via a natural-language prompt ───────────────────
user_prompt_B = """
I have another Python function related to reaction rates.
It computes the Arrhenius factor exp(-Ea/kT) for a given activation barrier.
Please add it to the system too.

def arrhenius_factor(barrier_eV: float, temperature_K: float) -> str:
    'Compute the Arrhenius rate factor exp(-Ea/kT) for a barrier in eV at given temperature.'
    import math
    kT = 8.617333e-5 * temperature_K
    factor = math.exp(-barrier_eV / kT)
    return f'Arrhenius factor for Ea={barrier_eV} eV at {temperature_K:.1f} K = {factor:.6e}'
"""

result_B = register_tools_from_prompt(user_prompt_B, pool, model)
print(f"Registration : {result_B}")
print(f"Registry now : {pool.list_registered_tools()}")
assert "arrhenius_factor" in pool.list_registered_tools(), "arrhenius_factor not registered!"

Thread ID: e6f20a06-c502-4bf0-bd16-c4ca20645964
Registration : Already registered (skipped): arrhenius_factor
Registry now : ['boltzmann_energy', 'arrhenius_factor']


In [30]:
# ── Create thermal_agent and assign both tools via the supervisor pipeline ────
# The user describes what they want; PromptEnhancer routes to tool_manager.

create_prompt = (
    "I need a dedicated specialist for thermal physics calculations. "
    "Please create one and give it the two tools I just added: "
    "the Boltzmann energy calculator and the Arrhenius rate factor function."
)
config_setup = {'configurable': {'thread_id': 'setup'}}


print("── [enhancer] " + "─" * 44)
enhanced = enhancer.enhance(create_prompt)
print(enhanced)
print()

print("── [supervisor] " + "─" * 42)
print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': enhanced}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

── [enhancer] ────────────────────────────────────────────
I need a dedicated specialist for thermal physics calculations. Please create one and give it the two tools I just added: the Boltzmann energy calculator and the Arrhenius rate factor function. Use compute_agent — it should use boltzmann_energy and arrhenius_factor to compute the answer step by step.

── [supervisor] ──────────────────────────────────────────
Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_tool_manager

Successfully transferred to tool_manager

Update from node tool_manager:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The thermal physics specialist ag

In [31]:
# ── Verify state after Phase 2 ────────────────────────────────────────────────
# The LLM chooses the agent name; find it by excluding the static agents.
STATIC_AGENTS = {"shell_agent", "compute_agent"}
dynamic_agents = [a for a in pool.list_agents() if a not in STATIC_AGENTS]
assert len(dynamic_agents) == 1, f"Expected 1 dynamic agent, got: {dynamic_agents}"
SPECIALIST_NAME = dynamic_agents[0]
print(f"Dynamic agent created: '{SPECIALIST_NAME}'")

# Confirm both tools are assigned to the specialist
thermal_tools = [t.name for t in pool._agents[SPECIALIST_NAME]["extra_tools"]]
assert "boltzmann_energy" in thermal_tools, f"boltzmann_energy not assigned to {SPECIALIST_NAME}"
assert "arrhenius_factor" in thermal_tools, f"arrhenius_factor not assigned to {SPECIALIST_NAME}"

print_pool_state(pool, "AFTER PHASE 2 — ADD")

Dynamic agent created: 'thermal_physics_specialist'

  AFTER PHASE 2 — ADD
  Agents   : ['shell_agent', 'compute_agent', 'thermal_physics_specialist']
  Registry : ['boltzmann_energy', 'arrhenius_factor']

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)
  [thermal_physics_specialist]
    base tools    : (none)
    assigned tools: ['arrhenius_factor', 'boltzmann_energy']


---
## 3 — Use: Run Computations Through the Supervisor Pipeline

The user asks scientific questions using plain language.
The PromptEnhancer inspects the live pool and appends routing hints;
the supervisor dispatches to the thermal specialist which executes the tools.

| Query | Expected route | Tools called |
|-------|---------------|--------------|
| "What is kT at 300 K?" | → specialist agent | `boltzmann_energy` |
| "Arrhenius factor, Ea=0.3 eV, T=400 K?" | → specialist agent | `boltzmann_energy` + `arrhenius_factor` |

In [32]:
# ── Query 1: Single tool — Boltzmann energy at 300 K ─────────────────────────
query_1 = "What is the thermal energy kT at room temperature (300 K)?"

print("── [enhancer] " + "─" * 44)
enhanced_1 = enhancer.enhance(query_1)
print(enhanced_1)
print()

print("── [supervisor] " + "─" * 42)
print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': query_1}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

── [enhancer] ────────────────────────────────────────────
What is the thermal energy kT at room temperature (300 K)? Use thermal_physics_specialist — it should use boltzmann_energy to compute the answer step by step.

── [supervisor] ──────────────────────────────────────────
Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_thermal_physics_specialist

Successfully transferred to thermal_physics_specialist

Update from node thermal_physics_specialist:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The thermal energy \( kT \) at room temperature (300 K) is approximately 0.02585 eV.



In [33]:
# ── Query 2: Two tools — Arrhenius factor at 400 K, barrier 0.3 eV ───────────
# The agent should call boltzmann_energy (to get kT) and arrhenius_factor.
query_2 = (
    'A surface diffusion process on a metal catalyst has an activation barrier of 0.30 eV. '
    'What is the Arrhenius Boltzmann factor at 400 K?'
)

print("── [enhancer] " + "─" * 44)
enhanced_2 = enhancer.enhance(query_2)
print(enhanced_2)
print()

print("── [supervisor] " + "─" * 42)
print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': enhanced_2}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

── [enhancer] ────────────────────────────────────────────
A surface diffusion process on a metal catalyst has an activation barrier of 0.30 eV. What is the Arrhenius Boltzmann factor at 400 K? Use thermal_physics_specialist — it should use arrhenius_factor and boltzmann_energy to compute the answer step by step.

── [supervisor] ──────────────────────────────────────────
Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_thermal_physics_specialist

Successfully transferred to thermal_physics_specialist

Update from node thermal_physics_specialist:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The thermal energy \( kT \) at 400 K 

---
## 4 — Remove: Clean Up the Dynamic Agent and Tools

The user asks to remove the specialist and its tools.
The supervisor routes to `tool_manager`, which:
1. Removes `thermal_agent` from the pool (supervisor rebuilds automatically)
2. Unregisters `boltzmann_energy` and `arrhenius_factor` from the global registry

After this phase the system should be back to its initial state.

In [36]:
# ── Remove the dynamic specialist agent ──────────────────────────────────────
remove_agent_prompt = (
    f"The thermal physics specialist is no longer needed. "
    "Please remove it from the system."
)

print("── [enhancer] " + "─" * 44)
enhanced_remove = enhancer.enhance(remove_agent_prompt)
print(enhanced_remove)
print()

print("── [supervisor] " + "─" * 42)
print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': enhanced_2}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

── [enhancer] ────────────────────────────────────────────
The thermal physics specialist is no longer needed. Please remove it from the system.

── [supervisor] ──────────────────────────────────────────
Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_tool_manager

Successfully transferred to tool_manager

Update from node tool_manager:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The thermal physics specialist agent has been recreated and equipped with the Boltzmann energy calculator and Arrhenius rate factor function tools. Please provide the calculation request again, and it will compute the answer step by step.



In [39]:
# ── Remove both tools from the registry ───────────────────────────────────────
remove_tools_prompt = (
    "Also remove the two thermal calculation tools from the system — "
    "the Boltzmann energy one and the Arrhenius rate factor one."
)

print("── [enhancer] " + "─" * 44)
enhanced_rt = enhancer.enhance(remove_tools_prompt)
print(enhanced_rt)
print()

print("── [supervisor] " + "─" * 42)
print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': enhanced_2}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

registry_after = pool.list_registered_tools()
assert "boltzmann_energy" not in registry_after, "boltzmann_energy still in registry!"
assert "arrhenius_factor" not in registry_after, "arrhenius_factor still in registry!"
print(f"\n✓ Registry cleared: {registry_after or '(empty)'}")

── [enhancer] ────────────────────────────────────────────
Also remove the two thermal calculation tools from the system — the Boltzmann energy one and the Arrhenius rate factor one. Use tool_manager — it should use unregister_tool to remove these tools step by step.

── [supervisor] ──────────────────────────────────────────
Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_thermal_physics_specialist

Successfully transferred to thermal_physics_specialist

Update from node thermal_physics_specialist:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The thermal energy \( kT \) at 400 K is approximately 0.03447 eV.

Using this, the A

In [40]:
# ── Final state: confirm system is back to initial architecture ───────────────
final_agents   = pool.list_agents()
final_registry = pool.list_registered_tools()

assert set(final_agents) == STATIC_AGENTS, f"Unexpected agents remaining: {final_agents}"
assert final_registry == [], f"Registry not empty: {final_registry}"

print_pool_state(pool, "FINAL STATE — matches initial architecture")

AssertionError: Unexpected agents remaining: ['shell_agent', 'compute_agent', 'thermal_physics_specialist']